In [1]:
import sys
sys.path.append("/kaggle/input/library-metrics")

In [2]:
import os, time, psutil
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
from lightgbm import LGBMClassifier

from metric_toolkit import evaluate_all_metrics

FS_METHOD = "GA_SVM"

X_train = pd.read_csv("/kaggle/input/data-time-complexity/X_train.csv")
X_test = pd.read_csv("/kaggle/input/data-time-complexity/X_test.csv")

y_train = pd.read_csv("/kaggle/input/data-time-complexity/y_train.csv").values.ravel()
y_test = pd.read_csv("/kaggle/input/data-time-complexity/y_test.csv").values.ravel()

mask = np.load(f"/kaggle/input/fs-result/ga_mask.npy")
X_train_sel = X_train.loc[:, mask]
X_test_sel = X_test.loc[:, mask]

fs_metrics_path = f"/kaggle/input/fs-result/fs_metrics_ga.csv"
fs_metrics = pd.read_csv(fs_metrics_path).iloc[0]

wall_time = fs_metrics["WallTime(s)"]
cpu_util  = fs_metrics["CPUUtil(%)"]
peak_mem  = fs_metrics["PeakMem(MB)"]
base_mem  = fs_metrics["BaseMem(MB)"]
energy    = fs_metrics["Energy(J)"]
carbon    = fs_metrics["Carbon(gCO2e)"]
edp       = fs_metrics["EDP(J*s)"]
cache_hit = fs_metrics.get("CacheHit", np.nan)

models = {
    "KNN": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    "LogisticRegression": LogisticRegression(max_iter=200, solver='lbfgs', random_state=42),
    "NaiveBayes": GaussianNB(),
    "DecisionTree": DecisionTreeClassifier(max_depth=None, random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost": XGBClassifier(
        n_estimators=200, 
        learning_rate=0.1, 
        max_depth=6, 
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="mlogloss",
        use_label_encoder=False,
        random_state=42,
        n_jobs=-1
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    ),
    "SVM": SVC(C=0.1, kernel="linear", probability=True, random_state=42)
}

results = []
for name, model in models.items():
    pipeline = Pipeline([
        ("clf", model)
    ])
    pipeline.fit(X_train_sel, y_train)

    y_pred = pipeline.predict(X_test_sel)
    y_prob = pipeline.predict_proba(X_test_sel) if hasattr(pipeline, "predict_proba") else None

    metrics = evaluate_all_metrics(
        y_true=y_test,
        y_pred=y_pred,
        y_prob=y_prob,
        cpu_util=cpu_util,
        wall_time=wall_time,
        peak_mem_mb=peak_mem,
        base_mem_mb=base_mem,
        num_evals=X_train_sel.shape[1],
        missing_flags=None,
        labels_for_leak=y_test,
        cache_hit=cache_hit,
    )

    metrics.update({
        "FS_Method": FS_METHOD,
        "Classifier": name,
        "NumFeatures": X_train_sel.shape[1],
        "Energy(J)": energy,
        "Carbon(gCO2e)": carbon,
        "EDP(J*s)": edp,
        "Timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
    })
    results.append(metrics)
    print(f"{name}: Accuracy={metrics['Accuracy']:.4f}, MacroF1={metrics['MacroF1']:.4f}")

out_path = "/kaggle/working/classification_metrics_all.csv"
pd.DataFrame(results).to_csv(out_path, index=False)
print("Đã ghi toàn bộ classification metrics.")
print(f"→ Saved to {out_path}")

AUC computation failed: average must be one of ('macro', 'weighted', None) for multiclass problems
KNN: Accuracy=0.5030, MacroF1=0.4901


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


AUC computation failed: average must be one of ('macro', 'weighted', None) for multiclass problems
LogisticRegression: Accuracy=0.7690, MacroF1=0.7684
AUC computation failed: average must be one of ('macro', 'weighted', None) for multiclass problems
NaiveBayes: Accuracy=0.3725, MacroF1=0.3512
AUC computation failed: average must be one of ('macro', 'weighted', None) for multiclass problems
DecisionTree: Accuracy=0.5960, MacroF1=0.5957
AUC computation failed: average must be one of ('macro', 'weighted', None) for multiclass problems
RandomForest: Accuracy=0.7710, MacroF1=0.7681
AUC computation failed: average must be one of ('macro', 'weighted', None) for multiclass problems
XGBoost: Accuracy=0.8515, MacroF1=0.8507
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.120960 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 132460
[LightGBM] [Info] Number of data points in the train set: 16000, number of use

In [3]:
!pip install --quiet gspread gspread_dataframe oauth2client

In [4]:
import gspread
from gspread_dataframe import set_with_dataframe
from oauth2client.service_account import ServiceAccountCredentials

def connect_to_google_sheet(sheet_url, credentials_path="sheets-credentials.json"):
    scope = [
        "https://spreadsheets.google.com/feeds",
        "https://www.googleapis.com/auth/drive",
    ]
    creds = ServiceAccountCredentials.from_json_keyfile_name(credentials_path, scope)
    client = gspread.authorize(creds)
    sheet = client.open_by_url(sheet_url)
    return sheet.sheet1

def append_metrics_to_sheet(worksheet, metrics_df):
    existing = pd.DataFrame(worksheet.get_all_records())
    combined = pd.concat([existing, metrics_df], ignore_index=True)
    worksheet.clear()
    set_with_dataframe(worksheet, combined)
    print("Metrics appended successfully!")

SHEET_URL = "https://docs.google.com/spreadsheets/d/1zKdcBjfuTdeSnJ_8YAbwRv8KZdSL6EAyDOo7DClNeJg/edit?gid=0#gid=0"
CREDENTIAL_PATH = "/kaggle/input/api-json/time-complexity-474404-ba31fb86904e.json"

worksheet = connect_to_google_sheet(SHEET_URL, CREDENTIAL_PATH)
append_metrics_to_sheet(worksheet, pd.DataFrame(results))

Metrics appended successfully!
